# Gold layer
Indicator tables per university and year, built from the silver tables.
Full counting: a work shared by several universities counts once for each of them.

In [0]:
CREATE OR REPLACE TABLE workspace.openalex.gold_output_by_year
COMMENT 'Works per university and year by document group (full counting).'
AS
SELECT
  wu.university,
  w.year,
  COUNT_IF(w.type IN ('article', 'review')) AS articles_reviews,
  COUNT_IF(w.type = 'conference-paper')     AS conference_papers,
  COUNT(*)                                  AS total
FROM workspace.openalex.silver_work_universities AS wu
JOIN workspace.openalex.silver_works AS w
  ON w.work_id = wu.work_id
GROUP BY wu.university, w.year;

SELECT *
FROM workspace.openalex.gold_output_by_year
ORDER BY university, year;

In [0]:
CREATE OR REPLACE TABLE workspace.openalex.gold_open_access
COMMENT 'Open access share per university and year, total and by route (articles and reviews, full counting).'
AS
SELECT
  wu.university,
  w.year,
  COUNT(*)                                                         AS n_works,
  COUNT_IF(w.is_oa)                                                AS n_oa,
  ROUND(100 * COUNT_IF(w.is_oa) / COUNT(*), 1)                     AS pct_oa,
  ROUND(100 * COUNT_IF(w.oa_status = 'gold')    / COUNT(*), 1)     AS pct_gold,
  ROUND(100 * COUNT_IF(w.oa_status = 'diamond') / COUNT(*), 1)     AS pct_diamond,
  ROUND(100 * COUNT_IF(w.oa_status = 'hybrid')  / COUNT(*), 1)     AS pct_hybrid,
  ROUND(100 * COUNT_IF(w.oa_status = 'bronze')  / COUNT(*), 1)     AS pct_bronze,
  ROUND(100 * COUNT_IF(w.oa_status = 'green')   / COUNT(*), 1)     AS pct_green,
  ROUND(100 * COUNT_IF(w.oa_status = 'closed')  / COUNT(*), 1)     AS pct_closed
FROM workspace.openalex.silver_work_universities AS wu
JOIN workspace.openalex.silver_works AS w
  ON w.work_id = wu.work_id
WHERE w.type IN ('article', 'review')
GROUP BY wu.university, w.year;

SELECT *
FROM workspace.openalex.gold_open_access
ORDER BY university, year;

In [0]:
CREATE OR REPLACE TABLE workspace.openalex.gold_international_collaboration
COMMENT 'International collaboration per university and year, articles/reviews and conference papers separately (full counting).'
AS
WITH counts AS (
  SELECT
    wu.university,
    w.year,
    COUNT_IF(w.type IN ('article', 'review'))                        AS n_art_rev,
    COUNT_IF(w.type IN ('article', 'review') AND w.is_international) AS n_int_art_rev,
    COUNT_IF(w.type = 'conference-paper')                            AS n_conf,
    COUNT_IF(w.type = 'conference-paper' AND w.is_international)     AS n_int_conf
  FROM workspace.openalex.silver_work_universities AS wu
  JOIN workspace.openalex.silver_works AS w
    ON w.work_id = wu.work_id
  GROUP BY wu.university, w.year
)
SELECT
  university,
  year,
  n_art_rev,
  n_int_art_rev,
  ROUND(100 * n_int_art_rev / n_art_rev, 1) AS pct_int_art_rev,
  n_conf,
  n_int_conf,
  ROUND(100 * n_int_conf / n_conf, 1)       AS pct_int_conf
FROM counts;

SELECT *
FROM workspace.openalex.gold_international_collaboration
ORDER BY university, year;

In [0]:
CREATE OR REPLACE TABLE workspace.openalex.gold_citation_impact
COMMENT 'Citation impact per university and year (articles and reviews with FWCI, full counting). Recent years have a short citation window.'
AS
SELECT
  wu.university,
  w.year,
  COUNT(*)                                        AS n_works,
  ROUND(AVG(w.fwci), 2)                           AS mean_fwci,
  ROUND(MEDIAN(w.fwci), 2)                        AS median_fwci,
  ROUND(100 * COUNT_IF(w.is_top10) / COUNT(*), 1) AS pct_top10,
  ROUND(100 * COUNT_IF(w.is_top1)  / COUNT(*), 1) AS pct_top1
FROM workspace.openalex.silver_work_universities AS wu
JOIN workspace.openalex.silver_works AS w
  ON w.work_id = wu.work_id
WHERE w.type IN ('article', 'review')
  AND w.fwci IS NOT NULL
GROUP BY wu.university, w.year;

SELECT *
FROM workspace.openalex.gold_citation_impact
ORDER BY university, year;

In [0]:
CREATE OR REPLACE TABLE workspace.openalex.gold_galician_collaboration
COMMENT 'Articles and reviews co-authored by each pair of Galician universities, per year. A work shared by all three counts once for each pair.'
AS
SELECT
  a.university AS uni_a,
  b.university AS uni_b,
  w.year,
  COUNT(*)     AS n_shared
FROM workspace.openalex.silver_work_universities AS a
JOIN workspace.openalex.silver_work_universities AS b
  ON a.work_id = b.work_id
 AND a.university < b.university
JOIN workspace.openalex.silver_works AS w
  ON w.work_id = a.work_id
WHERE w.type IN ('article', 'review')
GROUP BY a.university, b.university, w.year;

SELECT *
FROM workspace.openalex.gold_galician_collaboration
ORDER BY uni_a, uni_b, year;